# 1. GENERAR DATASET

In [1]:
import pandas as pd
import glob
import os

ruta = "PV_MaximumPowerPredictor/*.csv"

dfs = []

for archivo in glob.glob(ruta):
    
    nombre = os.path.basename(archivo).replace(".csv","")
    
    partes = nombre.split("_")
    
    parque_id = partes[0]              # Cocoa / Golden / Eugene
    panel_id = "_".join(partes[1:])    # tipo de panel
    
    df = pd.read_csv(archivo)
    
    # eliminar timestamp
    df = df.drop(columns=["Time Stamp (local standard time) yyyy-mm-ddThh:mm:ss"], errors="ignore")
    
    # añadir columnas
    df["parque_id"] = parque_id
    df["panel_id"] = panel_id
    
    # mover columnas al inicio
    cols = ["parque_id","panel_id"] + [c for c in df.columns if c not in ["parque_id","panel_id"]]
    df = df[cols]
    
    dfs.append(df)

dataset_total = pd.concat(dfs, ignore_index=True)

print(dataset_total.shape)
dataset_total.head()

(1025599, 12)


,parque_id,panel_id,POA irradiance CMP22 pyranometer (W/m2),PV module back surface temperature (degC),Pmp (W),Dry bulb temperature (degC),Relative humidity (%RH),Atmospheric pressure (mb),Precipitation (mm) accumulated daily total,Direct normal irradiance (W/m2),Global horizontal irradiance (W/m2),Diffuse horizontal irradiance (W/m2)
0,Cocoa,aSiMicro03036_cleaned,20.2,19.3,1.9610,17.7,96.0,1009.8,20.3,0.1,21.9,22.2
1,Cocoa,aSiMicro03036_cleaned,35.8,19.5,3.7242,17.8,96.0,1009.7,20.3,0.1,38.4,39.0
2,Cocoa,aSiMicro03036_cleaned,20.2,19.5,1.9551,17.8,96.0,1009.9,20.3,0.1,20.1,20.3
3,Cocoa,aSiMicro03036_cleaned,20.6,19.5,2.0057,17.8,96.1,1009.8,20.3,0.1,21.4,21.8
4,Cocoa,aSiMicro03036_cleaned,29.5,19.5,3.1397,17.8,96.0,1009.9,20.6,0.1,29.3,29.5


# 2. EXPLORAR LOS DATOS

In [2]:
dataset_total["parque_id"].value_counts() # Comprobar num de parques y paneles

parque_id
Eugene    473348
Cocoa     421664
Golden    130587
Name: count, dtype: int64

In [3]:
dataset_total["panel_id"].nunique()

22

In [4]:
sorted(dataset_total["panel_id"].unique())

['CIGS1-001_cleaned',
 'CIGS39013_cleaned',
 'CIGS39017_cleaned',
 'CIGS8-001_cleaned',
 'CdTe75638_cleaned',
 'CdTe75669_cleaned',
 'HIT05662_cleaned',
 'HIT05667_cleaned',
 'aSiMicro03036_cleaned',
 'aSiMicro03038_cleaned',
 'aSiTandem72-46_cleaned',
 'aSiTandem90-31_cleaned',
 'aSiTriple28324_cleaned',
 'aSiTriple28325_cleaned',
 'mSi0166_cleaned',
 'mSi0188_cleaned',
 'mSi0247_cleaned',
 'mSi0251_cleaned',
 'mSi460A8_cleaned',
 'mSi460BB_cleaned',
 'xSi11246_cleaned',
 'xSi12922_cleaned']

In [5]:
pd.crosstab(dataset_total["parque_id"], dataset_total["panel_id"])

panel_id,CIGS1-001_cleaned,CIGS39013_cleaned,CIGS39017_cleaned,CIGS8-001_cleaned,CdTe75638_cleaned,CdTe75669_cleaned,HIT05662_cleaned,HIT05667_cleaned,aSiMicro03036_cleaned,aSiMicro03038_cleaned,...,aSiTriple28324_cleaned,aSiTriple28325_cleaned,mSi0166_cleaned,mSi0188_cleaned,mSi0247_cleaned,mSi0251_cleaned,mSi460A8_cleaned,mSi460BB_cleaned,xSi11246_cleaned,xSi12922_cleaned
parque_id,,,,,,,,,,,,,,,,,,,,,
Cocoa,0,0,34775,38939,39080,0,0,38377,39037,0,...,38485,0,36765,39102,0,0,38929,0,0,38989
Eugene,0,0,42674,43146,42248,0,0,43271,43343,0,...,42705,0,43268,43127,0,0,43115,0,0,43185
Golden,12011,11437,0,0,0,11953,11876,0,0,12148,...,0,11445,0,0,11912,11887,0,11919,11929,0


In [6]:
dataset_total.isna().sum()

parque_id                                     0
panel_id                                      0
POA irradiance CMP22 pyranometer (W/m2)       0
PV module back surface temperature (degC)     0
Pmp (W)                                       0
Dry bulb temperature (degC)                   0
Relative humidity (%RH)                       0
Atmospheric pressure (mb)                     0
Precipitation (mm) accumulated daily total    0
Direct normal irradiance (W/m2)               0
Global horizontal irradiance (W/m2)           0
Diffuse horizontal irradiance (W/m2)          0
dtype: int64

# 3. MODELO BASELINE

No ignoramos el parque y el panel, solo el timestamp.

In [7]:
from sklearn.preprocessing import LabelEncoder

le_parque = LabelEncoder()
le_panel  = LabelEncoder()

dataset_total["parque_enc"] = le_parque.fit_transform(dataset_total["parque_id"])
dataset_total["panel_enc"]  = le_panel.fit_transform(dataset_total["panel_id"])

In [8]:
df_model = dataset_total.drop(columns=["parque_id", "panel_id"])

In [9]:
X = df_model.drop(columns=["Pmp (W)"])
y = df_model["Pmp (W)"]

In [10]:
from sklearn.model_selection import train_test_split

# Primero separar test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Luego separar val del resto (20% del total = 25% de train_val)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

# Resultado: 60% train, 20% val, 20% test
print(X_train.shape, X_val.shape, X_test.shape)

(615359, 11) (205120, 11) (205120, 11)


In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.fit_transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [12]:
import numpy as np

np.mean(X_train_scaled, axis=0)


array([ 1.00041476e-16, -2.15324728e-16, -1.34866625e-17,  1.27476673e-17,
       -2.12461122e-18, -7.57470086e-18,  2.47563394e-17,  1.30247905e-17,
       -8.82175527e-18,  7.84258662e-17,  1.85672545e-17])

In [13]:
np.std(X_train_scaled, axis=0)

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [14]:
import torch

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

## A. DISEÑO DEL MODELO

In [15]:
import torch
import torch.nn as nn

class PVModel(nn.Module):
    
    def __init__(self):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(11, 64),
            nn.ReLU(),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            
            nn.Linear(32, 16),
            nn.ReLU(),
            
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        return self.model(x)

In [16]:
class PVModelV2(nn.Module):
    
    def __init__(self, input_size=11):
        super().__init__()
        
        self.model = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        return self.model(x)

In [17]:
model = PVModel()
print(model)

model2 = PVModelV2()
print(model2)

PVModel(
  (model): Sequential(
    (0): Linear(in_features=11, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): ReLU()
    (6): Linear(in_features=16, out_features=1, bias=True)
  )
)
PVModelV2(
  (model): Sequential(
    (0): Linear(in_features=11, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=32, out_features=16, bias=True)
    (9): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_feature

In [18]:
model(X_train_tensor[:5])

tensor([[0.2450],
        [0.2371],
        [0.2393],
        [0.2379],
        [0.2391]], grad_fn=<AddmmBackward0>)

In [19]:
model2(X_train_tensor[:5])

tensor([[-0.3556],
        [ 1.4632],
        [ 0.7080],
        [ 0.0247],
        [ 0.4426]], grad_fn=<AddmmBackward0>)

In [20]:
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [21]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1024
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1024
)

## B. ML FLOW

In [22]:
import mlflow

In [23]:
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("PV_Power")

c:\Users\libeb\OneDrive\Documentos\GitHub\Reto3_MicroRedes\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
Traceback (most recent call last):
  File "c:\Users\libeb\OneDrive\Documentos\GitHub\Reto3_MicroRedes\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\libeb\OneDrive\Documentos\GitHub\Reto3_MicroRedes\.venv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileS

<Experiment: artifact_location='file:c:/Users/libeb/OneDrive/Documentos/GitHub/Reto3_MicroRedes/OBJETIVO2/mlruns/542888792710679087', creation_time=1774084520213, experiment_id='542888792710679087', last_update_time=1774084520213, lifecycle_stage='active', name='PV_Power', tags={}, workspace='default'>

In [25]:
class PVModel(nn.Module):
    
    def __init__(self, layers_sizes, input_size=11):
        super().__init__()
        
        layers = []
        
        for size in layers_sizes:
            layers.append(nn.Linear(input_size, size))
            layers.append(nn.ReLU())
            input_size = size
        
        layers.append(nn.Linear(input_size, 1))
        
        self.model = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.model(x)

In [26]:
class PVModelV2(nn.Module):
    
    def __init__(self, layers_sizes,dropout, input_size=11):
        super().__init__()
        
        layers = []
        
        for size in layers_sizes:
            layers.append(nn.Linear(input_size, size))
            layers.append(nn.BatchNorm1d(size))   # nuevo
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))         # nuevo
            input_size = size
        
        layers.append(nn.Linear(input_size, 1))
        
        self.model = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.model(x)

In [27]:
def evaluate(model, loader, criterion):
    
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
    
    return total_loss / len(loader)

In [28]:
def train_model(arch, lr, epochs, model,dropout=0):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float("inf")
    patience = 5
    counter = 0
    
    with mlflow.start_run():
    
        mlflow.log_param("arquitectura", str(arch))
        mlflow.log_param("lr", lr)
        mlflow.log_param("dropout",dropout)
        for epoch in range(epochs):
            
            model.train()
            total_loss = 0
            
            for X_batch, y_batch in train_loader:
                
                y_pred = model(X_batch)
                loss = criterion(y_pred, y_batch)
                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            train_loss = total_loss / len(train_loader)
            val_loss = evaluate(model, val_loader, criterion)
            
            mlflow.log_metric("train_loss", train_loss, step=epoch)
            mlflow.log_metric("val_loss", val_loss, step=epoch)
            
            print(f"Epoch {epoch+1} | Train: {train_loss:.2f} | Val: {val_loss:.2f}")

            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), f"best_model.pt")
                counter = 0
            else:
                counter += 1
            if counter >= patience:
                print(f"Early stopping en epoch {epoch+1}")
                break

    return model

In [29]:
arquitecturas = [
    [64, 32, 16],
    [128, 64, 32],
    [256, 128, 64]
]

learning_rates = [0.001, 0.0005]

dropouts = [0.1, 0.2]

epochs = 100

In [ ]:
for arch in arquitecturas:
    for lr in learning_rates:
        model = PVModel(arch)
        print(f"\nEntrenando modelo {arch} con lr={lr}")
        
        train_model(arch, lr, epochs,model)


Entrenando modelo [64, 32, 16] con lr=0.001
Epoch 1 | Train: 875.12 | Val: 506.08
Epoch 2 | Train: 351.19 | Val: 242.30
Epoch 3 | Train: 193.53 | Val: 154.02
Epoch 4 | Train: 129.81 | Val: 111.18
Epoch 5 | Train: 99.30 | Val: 95.68
Epoch 6 | Train: 83.32 | Val: 80.86
Epoch 7 | Train: 73.16 | Val: 69.75
Epoch 8 | Train: 66.23 | Val: 66.24
Epoch 9 | Train: 61.42 | Val: 58.92
Epoch 10 | Train: 54.17 | Val: 51.01
Epoch 11 | Train: 48.65 | Val: 47.44
Epoch 12 | Train: 45.30 | Val: 43.13
Epoch 13 | Train: 41.88 | Val: 41.51
Epoch 14 | Train: 38.56 | Val: 36.45
Epoch 15 | Train: 35.60 | Val: 33.96
Epoch 16 | Train: 32.98 | Val: 31.54
Epoch 17 | Train: 31.02 | Val: 30.04
Epoch 18 | Train: 29.22 | Val: 28.87
Epoch 19 | Train: 27.89 | Val: 28.08
Epoch 20 | Train: 26.47 | Val: 24.71
Epoch 21 | Train: 25.02 | Val: 26.00
Epoch 22 | Train: 23.67 | Val: 23.73
Epoch 23 | Train: 22.79 | Val: 24.59
Epoch 24 | Train: 22.00 | Val: 21.97
Epoch 25 | Train: 21.19 | Val: 20.59
Epoch 26 | Train: 20.65 | Val: 

In [ ]:
for arch in arquitecturas: 
    for lr in learning_rates:
        model = PVModelV2(arch,dropout=dropouts)
        print(f"\nEntrenando modelo {arch} con lr={lr}")
        train_model(arch, lr, epochs,model)


Entrenando modelo [64, 32, 16] con lr=0.001
Epoch 1 | Train: 1832.45 | Val: 945.68
Epoch 2 | Train: 430.07 | Val: 129.97
Epoch 3 | Train: 198.07 | Val: 75.97
Epoch 4 | Train: 180.36 | Val: 65.86
Epoch 5 | Train: 167.48 | Val: 59.71
Epoch 6 | Train: 159.62 | Val: 51.52
Epoch 7 | Train: 152.91 | Val: 50.32
Epoch 8 | Train: 148.05 | Val: 38.20
Epoch 9 | Train: 142.65 | Val: 44.38
Epoch 10 | Train: 139.12 | Val: 36.93
Epoch 11 | Train: 137.75 | Val: 37.81
Epoch 12 | Train: 134.73 | Val: 30.48
Epoch 13 | Train: 133.75 | Val: 32.84
Epoch 14 | Train: 131.86 | Val: 33.62
Epoch 15 | Train: 127.49 | Val: 38.22
Epoch 16 | Train: 129.71 | Val: 34.14
Epoch 17 | Train: 127.42 | Val: 26.56
Epoch 18 | Train: 128.06 | Val: 31.90
Epoch 19 | Train: 124.44 | Val: 25.10
Epoch 20 | Train: 125.22 | Val: 29.10
Epoch 21 | Train: 122.50 | Val: 30.67
Epoch 22 | Train: 123.23 | Val: 29.46
Epoch 23 | Train: 120.21 | Val: 26.58
Epoch 24 | Train: 120.61 | Val: 26.08
Early stopping en epoch 24

Entrenando modelo [64

# MODELOS SIMPLES

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Entrenar modelo
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predicción
y_pred = lr_model.predict(X_val)

# Métricas
mse = mean_squared_error(y_val, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_val, y_pred)

print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

MSE: 576.3045725256211
RMSE: 24.006344422373456
R2: 0.5934330172716631


In [ ]:
y_pred_nn = model(X_val_tensor).detach().numpy()

In [ ]:
mean_squared_error(y_val, y_pred_nn)

2.7023941642793217

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Modelo
model = XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Entrenar
model.fit(X_train, y_train)

# Predecir
y_pred = model.predict(X_val)

# Métricas
mse = mean_squared_error(y_val, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_val, y_pred)

print("MSE:", mse)
print("RMSE:", rmse)
print("R2:", r2)

MSE: 2.490038629218615
RMSE: 1.5779856238947854
R2: 0.9982433464167709


# 4. FEATURE ENGENIERING

In [ ]:
corr = df_model.corr(numeric_only=True)
corr["Pmp (W)"].sort_values(ascending=False)

Pmp (W)                                       1.000000
POA irradiance CMP22 pyranometer (W/m2)       0.748333
PV module back surface temperature (degC)     0.561800
Direct normal irradiance (W/m2)               0.068805
Global horizontal irradiance (W/m2)           0.057096
Precipitation (mm) accumulated daily total   -0.018312
Dry bulb temperature (degC)                  -0.019508
Diffuse horizontal irradiance (W/m2)         -0.023923
Atmospheric pressure (mb)                    -0.031497
Relative humidity (%RH)                      -0.033575
parque_enc                                   -0.036538
panel_enc                                    -0.179015
Name: Pmp (W), dtype: float64

In [ ]:
df_model["physical_model"] = (
    df_model["POA irradiance CMP22 pyranometer (W/m2)"] *
    (1 - 0.004 * (df_model["PV module back surface temperature (degC)"] - 25))
)

In [ ]:
df_model = df_model[[
    "POA irradiance CMP22 pyranometer (W/m2)",
    "PV module back surface temperature (degC)",
    "physical_model",
    "Pmp (W)"
]]

In [ ]:
X = df_model.drop(columns=["Pmp (W)"])
y = df_model["Pmp (W)"]

In [ ]:
from sklearn.model_selection import train_test_split

# Primero separar test (20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Luego separar val del resto (20% del total = 25% de train_val)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

# Resultado: 60% train, 20% val, 20% test
print(X_train.shape, X_val.shape, X_test.shape)

(615359, 3) (205120, 3) (205120, 3)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.fit_transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1024,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1024
)

In [ ]:
for arch in arquitecturas:
    for lr in learning_rates:
        model = PVModel(arch)
        print(f"\nEntrenando modelo {arch} con lr={lr}")
        train_model(arch, lr, epochs,model)


Entrenando modelo [64, 32, 16] con lr=0.001
Epoch 1 | Train: 783.75 | Val: 440.72
Epoch 2 | Train: 298.80 | Val: 185.69
Epoch 3 | Train: 144.22 | Val: 113.72
Epoch 4 | Train: 94.83 | Val: 82.66
Epoch 5 | Train: 67.27 | Val: 61.18
Epoch 6 | Train: 49.45 | Val: 42.40
Epoch 7 | Train: 36.00 | Val: 31.09
Epoch 8 | Train: 25.44 | Val: 20.71
Epoch 9 | Train: 17.92 | Val: 14.90
Epoch 10 | Train: 13.23 | Val: 11.03
Epoch 11 | Train: 10.40 | Val: 9.97
Epoch 12 | Train: 8.58 | Val: 7.54
Epoch 13 | Train: 7.25 | Val: 7.47
Epoch 14 | Train: 6.36 | Val: 5.68
Epoch 15 | Train: 5.59 | Val: 4.94
Epoch 16 | Train: 4.84 | Val: 4.48
Epoch 17 | Train: 4.36 | Val: 3.77
Epoch 18 | Train: 3.82 | Val: 3.83
Epoch 19 | Train: 3.44 | Val: 3.62
Epoch 20 | Train: 3.29 | Val: 3.21
Epoch 21 | Train: 3.05 | Val: 3.24
Epoch 22 | Train: 3.00 | Val: 2.76
Epoch 23 | Train: 2.85 | Val: 3.02
Epoch 24 | Train: 2.65 | Val: 2.79
Epoch 25 | Train: 2.60 | Val: 2.64
Epoch 26 | Train: 2.58 | Val: 2.96
Epoch 27 | Train: 2.45 | Va

KeyboardInterrupt: 

In [ ]:
for arch in arquitecturas:
    for lr in learning_rates:
        model = PVModelV2(arch,dropout=dropouts)
        print(f"\nEntrenando modelo {arch} con lr={lr}")
        train_model(arch, lr, epochs,model)

# ENTRENAMIENTOS FINALES DE LOS MEJORES MODELOS